# Notebook 03. Part 2: Validation Against the Official Amazon Reviews 2023 Electronics 5-Core Benchmark (Earliest Duplicate Policy)

This notebook validates the independently implemented preprocessing pipeline against the official **Amazon Reviews 2023 Electronics 5-core `rating_only` benchmark** after modifying the preprocessing pipeline to retain the **earliest** review for duplicate user–product pairs.

The comparison focuses on:

1. schema and data types;
2. numbers of interactions, users, and products;
3. equality of retained user–product pairs;
4. equality of ratings and timestamps for matching pairs;
5. verification that the implemented duplicate-review policy reproduces the official benchmark.

The official benchmark documentation states that `rating_only` contains the columns `user_id`, `parent_asin`, `rating`, and `timestamp`, and that repeated reviews for the same user–product pair are resolved by retaining the earliest review before iterative k-core filtering.

This notebook uses DuckDB to compare the official benchmark with the independently generated preprocessing output without loading the complete datasets into memory.

## 1. Imports and configuration

The path-resolution cell searches both the current working directory and its parent directories. This allows the notebook to run whether it is opened from the repository root or from a `notebooks/` directory.

In [1]:
from pathlib import Path
import gzip
import json
import time

import duckdb
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root by locating the expected data directories."""
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Expected a 'data/processed' directory "
        "in the current directory or one of its parents."
    )


PROJECT_ROOT = find_project_root()
OFFICIAL_PATH = PROJECT_ROOT / "data" / "benchmark" / "Electronics.csv.gz"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "interactions_final_earliest" / "interactions.parquet"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "benchmark_validation_earliest"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root:      {PROJECT_ROOT}")
print(f"Official dataset:  {OFFICIAL_PATH}")
print(f"Processed dataset: {PROCESSED_PATH}")
print(f"Validation output: {OUTPUT_DIR}")

assert OFFICIAL_PATH.exists(), f"Official benchmark not found: {OFFICIAL_PATH}"
assert PROCESSED_PATH.exists(), f"Processed dataset not found: {PROCESSED_PATH}"

Project root:      C:\Projects\gnn-recommender-dissertation
Official dataset:  C:\Projects\gnn-recommender-dissertation\data\benchmark\Electronics.csv.gz
Processed dataset: C:\Projects\gnn-recommender-dissertation\data\processed\interactions_final_earliest\interactions.parquet
Validation output: C:\Projects\gnn-recommender-dissertation\data\processed\benchmark_validation_earliest


## 2. Inspect the official compressed file

The released benchmark may be distributed either with or without a CSV header. The next cell inspects the first decompressed line and handles either form safely.

In [3]:
with gzip.open(OFFICIAL_PATH, "rt", encoding="utf-8") as file:
    first_line = file.readline().strip()

expected_header = "user_id,parent_asin,rating,timestamp"
official_has_header = first_line.lower() == expected_header

print("First line:")
print(first_line[:200])
print(f"\nHeader detected: {official_has_header}")

First line:
user_id,parent_asin,rating,timestamp

Header detected: True


## 3. Register both datasets as DuckDB views

Both sources are normalised to the same four comparison columns:

- `user_id`
- `parent_asin`
- `rating`
- `timestamp`

`verified_purchase` and any other locally retained fields are excluded because they are not included in the official `rating_only` benchmark.

In [4]:
connection = duckdb.connect(database=":memory:")

processed_path_sql = PROCESSED_PATH.as_posix().replace("'", "''")
official_path_sql = OFFICIAL_PATH.as_posix().replace("'", "''")

connection.execute(f"""
CREATE OR REPLACE VIEW ours AS
SELECT
    CAST(user_id AS VARCHAR) AS user_id,
    CAST(parent_asin AS VARCHAR) AS parent_asin,
    CAST(rating AS DOUBLE) AS rating,
    CAST(timestamp AS BIGINT) AS timestamp
FROM read_parquet('{processed_path_sql}');
""")

if official_has_header:
    connection.execute(f"""
    CREATE OR REPLACE VIEW official AS
    SELECT
        CAST(user_id AS VARCHAR) AS user_id,
        CAST(parent_asin AS VARCHAR) AS parent_asin,
        CAST(rating AS DOUBLE) AS rating,
        CAST(timestamp AS BIGINT) AS timestamp
    FROM read_csv(
        '{official_path_sql}',
        header = true,
        compression = 'gzip',
        columns = {{
            'user_id': 'VARCHAR',
            'parent_asin': 'VARCHAR',
            'rating': 'DOUBLE',
            'timestamp': 'BIGINT'
        }}
    );
    """)
else:
    connection.execute(f"""
    CREATE OR REPLACE VIEW official AS
    SELECT
        CAST(user_id AS VARCHAR) AS user_id,
        CAST(parent_asin AS VARCHAR) AS parent_asin,
        CAST(rating AS DOUBLE) AS rating,
        CAST(timestamp AS BIGINT) AS timestamp
    FROM read_csv(
        '{official_path_sql}',
        header = false,
        compression = 'gzip',
        columns = {{
            'user_id': 'VARCHAR',
            'parent_asin': 'VARCHAR',
            'rating': 'DOUBLE',
            'timestamp': 'BIGINT'
        }}
    );
    """)

print("Views created successfully.")

Views created successfully.


## 4. Schema inspection

This verifies that both datasets can be interpreted using compatible types before equality checks are performed.

In [5]:
print("Official schema")
display(connection.execute("DESCRIBE official").df())

print("Local processed schema")
display(connection.execute("DESCRIBE ours").df())

Official schema


,column_name,column_type,null,key,default,extra
0,user_id,VARCHAR,YES,None,None,None
1,parent_asin,VARCHAR,YES,None,None,None
2,rating,DOUBLE,YES,None,None,None
3,timestamp,BIGINT,YES,None,None,None


Local processed schema


,column_name,column_type,null,key,default,extra
0,user_id,VARCHAR,YES,None,None,None
1,parent_asin,VARCHAR,YES,None,None,None
2,rating,DOUBLE,YES,None,None,None
3,timestamp,BIGINT,YES,None,None,None


## 5. Aggregate dataset statistics

Matching aggregate statistics are necessary but not sufficient. Two datasets could have the same counts while containing different user–product pairs, so pair-level reconciliation follows.

In [6]:
statistics_query = """
WITH official_stats AS (
    SELECT
        COUNT(*) AS interactions,
        COUNT(DISTINCT user_id) AS users,
        COUNT(DISTINCT parent_asin) AS items,
        COUNT(DISTINCT (user_id, parent_asin)) AS unique_pairs
    FROM official
),
our_stats AS (
    SELECT
        COUNT(*) AS interactions,
        COUNT(DISTINCT user_id) AS users,
        COUNT(DISTINCT parent_asin) AS items,
        COUNT(DISTINCT (user_id, parent_asin)) AS unique_pairs
    FROM ours
)
SELECT metric, official_value, our_value, official_value = our_value AS matches
FROM (
    SELECT 'Interactions' AS metric, o.interactions AS official_value, m.interactions AS our_value
    FROM official_stats o, our_stats m
    UNION ALL
    SELECT 'Users', o.users, m.users FROM official_stats o, our_stats m
    UNION ALL
    SELECT 'Items', o.items, m.items FROM official_stats o, our_stats m
    UNION ALL
    SELECT 'Unique user-item pairs', o.unique_pairs, m.unique_pairs
    FROM official_stats o, our_stats m
)
ORDER BY CASE metric
    WHEN 'Interactions' THEN 1
    WHEN 'Users' THEN 2
    WHEN 'Items' THEN 3
    ELSE 4
END;
"""

statistics_df = connection.execute(statistics_query).df()
display(statistics_df.style.format({"official_value": "{:,}", "our_value": "{:,}"}))

,metric,official_value,our_value,matches
0,Interactions,"15,473,536","15,473,536",True
1,Users,"1,641,026","1,641,026",True
2,Items,"368,228","368,228",True
3,Unique user-item pairs,"15,473,536","15,473,536",True


## 6. Internal uniqueness checks

Both datasets should contain at most one row for each `(user_id, parent_asin)` pair.

In [7]:
uniqueness_query = """
SELECT
    dataset,
    interactions,
    unique_pairs,
    interactions - unique_pairs AS additional_duplicate_rows,
    interactions = unique_pairs AS one_row_per_pair
FROM (
    SELECT 'Official benchmark' AS dataset,
           COUNT(*) AS interactions,
           COUNT(DISTINCT (user_id, parent_asin)) AS unique_pairs
    FROM official
    UNION ALL
    SELECT 'Local processed dataset' AS dataset,
           COUNT(*) AS interactions,
           COUNT(DISTINCT (user_id, parent_asin)) AS unique_pairs
    FROM ours
);
"""

uniqueness_df = connection.execute(uniqueness_query).df()
display(uniqueness_df.style.format({
    "interactions": "{:,}",
    "unique_pairs": "{:,}",
    "additional_duplicate_rows": "{:,}"
}))

,dataset,interactions,unique_pairs,additional_duplicate_rows,one_row_per_pair
0,Official benchmark,"15,473,536","15,473,536",0,True
1,Local processed dataset,"15,473,536","15,473,536",0,True


## 7. User–product pair reconciliation

This is the central structural validation.

- `official_only_pairs`: present in the official benchmark but absent locally;
- `ours_only_pairs`: present locally but absent from the official benchmark;
- `shared_pairs`: present in both datasets.

If both one-sided counts are zero, the graph contains exactly the same user–product edges even if retained ratings or timestamps differ.

In [8]:
start_time = time.perf_counter()

pair_reconciliation_query = """
WITH official_only AS (
    SELECT user_id, parent_asin FROM official
    EXCEPT
    SELECT user_id, parent_asin FROM ours
),
ours_only AS (
    SELECT user_id, parent_asin FROM ours
    EXCEPT
    SELECT user_id, parent_asin FROM official
),
shared AS (
    SELECT o.user_id, o.parent_asin
    FROM official o
    INNER JOIN ours m USING (user_id, parent_asin)
)
SELECT
    (SELECT COUNT(*) FROM official_only) AS official_only_pairs,
    (SELECT COUNT(*) FROM ours_only) AS ours_only_pairs,
    (SELECT COUNT(*) FROM shared) AS shared_pairs;
"""

pair_reconciliation_df = connection.execute(pair_reconciliation_query).df()
elapsed = time.perf_counter() - start_time

display(pair_reconciliation_df.style.format("{:,}"))
print(f"Pair reconciliation completed in {elapsed / 60:.2f} minutes.")

,official_only_pairs,ours_only_pairs,shared_pairs
0,0,0,"15,473,536"


Pair reconciliation completed in 0.14 minutes.


## 8. Rating and timestamp reconciliation for shared pairs

Shared pairs are classified into four mutually exclusive groups:

1. rating and timestamp both match;
2. rating matches but timestamp differs;
3. timestamp matches but rating differs;
4. both rating and timestamp differ.

A different duplicate-retention rule can preserve the same graph edge while retaining a different timestamp and possibly a different rating.

In [9]:
value_reconciliation_query = """
SELECT
    CASE
        WHEN o.rating = m.rating AND o.timestamp = m.timestamp
            THEN 'Rating and timestamp match'
        WHEN o.rating = m.rating AND o.timestamp <> m.timestamp
            THEN 'Rating matches; timestamp differs'
        WHEN o.rating <> m.rating AND o.timestamp = m.timestamp
            THEN 'Timestamp matches; rating differs'
        ELSE 'Rating and timestamp differ'
    END AS comparison_result,
    COUNT(*) AS pairs
FROM official o
INNER JOIN ours m USING (user_id, parent_asin)
GROUP BY 1
ORDER BY pairs DESC;
"""

value_reconciliation_df = connection.execute(value_reconciliation_query).df()
value_reconciliation_df["percentage_of_shared_pairs"] = (
    value_reconciliation_df["pairs"] / value_reconciliation_df["pairs"].sum() * 100
)

display(value_reconciliation_df.style.format({
    "pairs": "{:,}",
    "percentage_of_shared_pairs": "{:.6f}%"
}))

,comparison_result,pairs,percentage_of_shared_pairs
0,Rating and timestamp match,"15,473,535",99.999994%
1,Timestamp matches; rating differs,1,0.000006%


## 9. Direction of timestamp differences

If the official benchmark retains the earliest repeated review and the local pipeline retains the latest, affected local timestamps should normally be later than the official timestamps.


In [10]:
timestamp_direction_query = """
SELECT
    CASE
        WHEN m.timestamp > o.timestamp THEN 'Local timestamp is later'
        WHEN m.timestamp < o.timestamp THEN 'Local timestamp is earlier'
        ELSE 'Timestamps are equal'
    END AS timestamp_relationship,
    COUNT(*) AS pairs
FROM official o
INNER JOIN ours m USING (user_id, parent_asin)
GROUP BY 1
ORDER BY pairs DESC;
"""

timestamp_direction_df = connection.execute(timestamp_direction_query).df()
display(timestamp_direction_df.style.format({"pairs": "{:,}"}))

,timestamp_relationship,pairs
0,Timestamps are equal,"15,473,536"


## 10. Inspect a sample of differing records

This deterministic sample helps interpret rating and timestamp differences.

In [11]:
difference_sample_query = """
SELECT
    o.user_id,
    o.parent_asin,
    o.rating AS official_rating,
    m.rating AS local_rating,
    o.timestamp AS official_timestamp,
    m.timestamp AS local_timestamp,
    m.timestamp - o.timestamp AS timestamp_difference_ms
FROM official o
INNER JOIN ours m USING (user_id, parent_asin)
WHERE o.rating <> m.rating OR o.timestamp <> m.timestamp
ORDER BY o.user_id, o.parent_asin
LIMIT 20;
"""

difference_sample_df = connection.execute(difference_sample_query).df()
display(difference_sample_df)

,user_id,parent_asin,official_rating,local_rating,official_timestamp,local_timestamp,timestamp_difference_ms
0,AG62UD7Q67DBARLYYTGSWHASW33Q,B0BHWQ3H17,1.0,5.0,1601607533154,1601607533154,0


## 11. Save validation results

The reconciliation figures are written to disk so Notebook 04 can reference them without
re-running the comparison against the official files.

In [12]:
statistics_df.to_csv(OUTPUT_DIR / "aggregate_statistics.csv", index=False)
uniqueness_df.to_csv(OUTPUT_DIR / "uniqueness_checks.csv", index=False)
pair_reconciliation_df.to_csv(OUTPUT_DIR / "pair_reconciliation.csv", index=False)
value_reconciliation_df.to_csv(OUTPUT_DIR / "value_reconciliation.csv", index=False)
timestamp_direction_df.to_csv(OUTPUT_DIR / "timestamp_direction.csv", index=False)
difference_sample_df.to_csv(OUTPUT_DIR / "difference_sample.csv", index=False)

validation_summary = {
    "official_path": str(OFFICIAL_PATH.relative_to(PROJECT_ROOT)),
    "processed_path": str(PROCESSED_PATH.relative_to(PROJECT_ROOT)),
    "official_header_detected": bool(official_has_header),
    "aggregate_statistics": statistics_df.to_dict(orient="records"),
    "uniqueness_checks": uniqueness_df.to_dict(orient="records"),
    "pair_reconciliation": pair_reconciliation_df.to_dict(orient="records"),
    "value_reconciliation": value_reconciliation_df.to_dict(orient="records"),
    "timestamp_direction": timestamp_direction_df.to_dict(orient="records"),
}

with open(OUTPUT_DIR / "benchmark_validation_summary.json", "w", encoding="utf-8") as file:
    json.dump(validation_summary, file, indent=2)

print(f"Validation outputs saved to: {OUTPUT_DIR}")

Validation outputs saved to: C:\Projects\gnn-recommender-dissertation\data\processed\benchmark_validation_earliest


## 12. Automated validation assertions

The assertions distinguish between:

- **structural reproduction**: identical users, products, interactions, and user–product edges;
- **value-level reproduction**: identical retained ratings and timestamps.

In [13]:
stats_lookup = statistics_df.set_index("metric")

assert bool(stats_lookup.loc["Interactions", "matches"]), "Interaction counts differ."
assert bool(stats_lookup.loc["Users", "matches"]), "User counts differ."
assert bool(stats_lookup.loc["Items", "matches"]), "Item counts differ."
assert bool(stats_lookup.loc["Unique user-item pairs", "matches"]), "Unique pair counts differ."

official_only_pairs = int(pair_reconciliation_df.loc[0, "official_only_pairs"])
ours_only_pairs = int(pair_reconciliation_df.loc[0, "ours_only_pairs"])

assert official_only_pairs == 0, f"{official_only_pairs:,} official pairs are missing locally."
assert ours_only_pairs == 0, f"{ours_only_pairs:,} local pairs are absent officially."

differing_pairs = int(
    value_reconciliation_df.loc[
        value_reconciliation_df["comparison_result"] != "Rating and timestamp match",
        "pairs"
    ].sum()
)

print("Structural reproduction: PASSED")
print(f"Shared pairs with a rating and/or timestamp difference: {differing_pairs:,}")

if differing_pairs == 0:
    print("Value-level reproduction: PASSED")
else:
    print("Value-level reproduction: NOT IDENTICAL. Review the duplicate policy and sample.")

Structural reproduction: PASSED
Shared pairs with a rating and/or timestamp difference: 1
Value-level reproduction: NOT IDENTICAL. Review the duplicate policy and sample.


## 13. Automated verification against the raw review data

The structural comparison shows that both datasets contain exactly the same user–product pairs. Because the retention policy now matches the benchmark, no pair differs by timestamp, and the raw-data verification below therefore finds no candidates to inspect. The section is retained for symmetry with the original run, where it identified the cause of the differences.

This section verifies the explanation directly against the original raw Electronics review file.

For a deterministic sample of differing pairs, it checks whether:

1. the official benchmark record exists among the earliest raw reviews for the pair; and
2. the local processed record exists among the latest raw reviews for the pair.

The raw dataset is scanned once and joined against the selected pairs. This is more efficient than running a separate raw-data query for each user–product pair.

In [14]:
RAW_REVIEWS_PATH = PROJECT_ROOT / "data" / "raw" / "Electronics.jsonl.gz"

assert RAW_REVIEWS_PATH.exists(), (
    f"Raw Electronics review file not found: {RAW_REVIEWS_PATH}"
)

# Start with 10 for a quick representative verification.
# Increase this number later, or set it to None to verify every differing pair.
RAW_VERIFICATION_LIMIT = 10

print(f"Raw reviews: {RAW_REVIEWS_PATH}")
print(f"Pairs selected for verification: {RAW_VERIFICATION_LIMIT}")

Raw reviews: C:\Projects\gnn-recommender-dissertation\data\raw\Electronics.jsonl.gz
Pairs selected for verification: 10


In [15]:
# Select the different pairs

connection.execute("DROP TABLE IF EXISTS verification_pairs")

limit_clause = (
    f"LIMIT {RAW_VERIFICATION_LIMIT}"
    if RAW_VERIFICATION_LIMIT is not None
    else ""
)

connection.execute(f"""
CREATE TEMP TABLE verification_pairs AS
SELECT
    o.user_id,
    o.parent_asin,
    o.rating AS official_rating,
    o.timestamp AS official_timestamp,
    m.rating AS local_rating,
    m.timestamp AS local_timestamp
FROM official o
INNER JOIN ours m
    USING (user_id, parent_asin)
WHERE o.timestamp <> m.timestamp
ORDER BY o.user_id, o.parent_asin
{limit_clause};
""")

selected_pairs_df = connection.execute("""
SELECT *
FROM verification_pairs
ORDER BY user_id, parent_asin;
""").df()

print(f"Selected pairs: {len(selected_pairs_df):,}")
display(selected_pairs_df)

Selected pairs: 0


,user_id,parent_asin,official_rating,official_timestamp,local_rating,local_timestamp


In [16]:
raw_reviews_path_sql = RAW_REVIEWS_PATH.as_posix().replace("'", "''")

start_time = time.perf_counter()

# AI-assisted: Claude helped refine this SQL verification query.

raw_verification_query = f"""
WITH matching_raw_reviews AS (
    SELECT
        CAST(r.user_id AS VARCHAR) AS user_id,
        CAST(r.parent_asin AS VARCHAR) AS parent_asin,
        CAST(r.rating AS DOUBLE) AS raw_rating,
        CAST(r.timestamp AS BIGINT) AS raw_timestamp
    FROM read_json_auto(
        '{raw_reviews_path_sql}',
        format = 'newline_delimited',
        compression = 'gzip'
    ) r
    INNER JOIN verification_pairs v
        ON CAST(r.user_id AS VARCHAR) = v.user_id
       AND CAST(r.parent_asin AS VARCHAR) = v.parent_asin
),

raw_pair_bounds AS (
    SELECT
        user_id,
        parent_asin,
        COUNT(*) AS raw_review_count,
        MIN(raw_timestamp) AS earliest_raw_timestamp,
        MAX(raw_timestamp) AS latest_raw_timestamp
    FROM matching_raw_reviews
    GROUP BY user_id, parent_asin
),

verification AS (
    SELECT
        v.user_id,
        v.parent_asin,

        v.official_rating,
        v.official_timestamp,

        v.local_rating,
        v.local_timestamp,

        b.raw_review_count,
        b.earliest_raw_timestamp,
        b.latest_raw_timestamp,

        BOOL_OR(
            r.raw_timestamp = b.earliest_raw_timestamp
            AND r.raw_rating = v.official_rating
        ) AS official_matches_earliest_raw_review,

        BOOL_OR(
            r.raw_timestamp = b.latest_raw_timestamp
            AND r.raw_rating = v.local_rating
        ) AS local_matches_latest_raw_review

    FROM verification_pairs v

    LEFT JOIN raw_pair_bounds b
        USING (user_id, parent_asin)

    LEFT JOIN matching_raw_reviews r
        USING (user_id, parent_asin)

    GROUP BY
        v.user_id,
        v.parent_asin,
        v.official_rating,
        v.official_timestamp,
        v.local_rating,
        v.local_timestamp,
        b.raw_review_count,
        b.earliest_raw_timestamp,
        b.latest_raw_timestamp
)

SELECT
    *,
    official_timestamp = earliest_raw_timestamp
        AS official_timestamp_is_earliest,

    local_timestamp = latest_raw_timestamp
        AS local_timestamp_is_latest,

    (
        official_timestamp = earliest_raw_timestamp
        AND official_matches_earliest_raw_review
    ) AS official_policy_verified,

    (
        local_timestamp = latest_raw_timestamp
        AND local_matches_latest_raw_review
    ) AS local_policy_verified

FROM verification
ORDER BY user_id, parent_asin;
"""

raw_verification_df = connection.execute(raw_verification_query).df()

elapsed = time.perf_counter() - start_time

display(raw_verification_df)

print(
    f"Raw verification completed in "
    f"{elapsed / 60:.2f} minutes."
)

,user_id,parent_asin,official_rating,official_timestamp,local_rating,local_timestamp,raw_review_count,earliest_raw_timestamp,latest_raw_timestamp,official_matches_earliest_raw_review,local_matches_latest_raw_review,official_timestamp_is_earliest,local_timestamp_is_latest,official_policy_verified,local_policy_verified


Raw verification completed in 0.02 minutes.


In [17]:
raw_verification_summary = pd.DataFrame({
    "check": [
        "Pairs selected",
        "Pairs found in raw data",
        "Official record verified as earliest",
        "Local record verified as latest",
        "Both policies verified"
    ],
    "pairs": [
        len(raw_verification_df),

        raw_verification_df[
            "raw_review_count"
        ].notna().sum(),

        raw_verification_df[
            "official_policy_verified"
        ].fillna(False).sum(),

        raw_verification_df[
            "local_policy_verified"
        ].fillna(False).sum(),

        (
            raw_verification_df[
                "official_policy_verified"
            ].fillna(False)
            &
            raw_verification_df[
                "local_policy_verified"
            ].fillna(False)
        ).sum()
    ]
})

raw_verification_summary["all_selected_pairs_pass"] = (
    raw_verification_summary["pairs"]
    == len(raw_verification_df)
)

display(raw_verification_summary)

,check,pairs,all_selected_pairs_pass
0,Pairs selected,0,True
1,Pairs found in raw data,0,True
2,Official record verified as earliest,0,True
3,Local record verified as latest,0,True
4,Both policies verified,0,True


No pairs were selected, since no timestamp differences remain. The check is vacuously satisfied and is reported here to show that the condition it tests no longer occurs.

In [18]:
expected_pairs = len(raw_verification_df)

assert expected_pairs > 0, (
    "No differing pairs were selected for raw verification."
)

assert raw_verification_df["raw_review_count"].notna().all(), (
    "At least one selected pair was not found in the raw review file."
)

assert (
    raw_verification_df["raw_review_count"] >= 2
).all(), (
    "At least one differing pair has fewer than two raw reviews."
)

assert raw_verification_df[
    "official_policy_verified"
].fillna(False).all(), (
    "At least one official benchmark record does not match "
    "the earliest raw review."
)

assert raw_verification_df[
    "local_policy_verified"
].fillna(False).all(), (
    "At least one local record does not match the latest raw review."
)

print("Raw duplicate-policy verification: PASSED")
print(
    f"All {expected_pairs:,} selected pairs confirm that:"
)
print("• the official benchmark retained the earliest review")
print("• the local preprocessing retained the latest review")

AssertionError: No differing pairs were selected for raw verification.

In [19]:
raw_history_query = f"""
SELECT
    CAST(r.user_id AS VARCHAR) AS user_id,
    CAST(r.parent_asin AS VARCHAR) AS parent_asin,
    CAST(r.rating AS DOUBLE) AS raw_rating,
    CAST(r.timestamp AS BIGINT) AS raw_timestamp,

    CASE
        WHEN CAST(r.timestamp AS BIGINT) = v.official_timestamp
             AND CAST(r.rating AS DOUBLE) = v.official_rating
            THEN 'Retained by official benchmark'

        WHEN CAST(r.timestamp AS BIGINT) = v.local_timestamp
             AND CAST(r.rating AS DOUBLE) = v.local_rating
            THEN 'Retained by local pipeline'

        ELSE 'Other duplicate review'
    END AS retention_status

FROM read_json_auto(
    '{raw_reviews_path_sql}',
    format = 'newline_delimited',
    compression = 'gzip'
) r

INNER JOIN verification_pairs v
    ON CAST(r.user_id AS VARCHAR) = v.user_id
   AND CAST(r.parent_asin AS VARCHAR) = v.parent_asin

ORDER BY
    user_id,
    parent_asin,
    raw_timestamp;
"""

raw_history_df = connection.execute(raw_history_query).df()

display(raw_history_df)

,user_id,parent_asin,raw_rating,raw_timestamp,retention_status


In [20]:
if (
    raw_verification_df.empty
    and raw_verification_summary.empty
    and raw_history_df.empty
):
    print("No raw verification differences found.")
else:
    raw_verification_df.to_csv(
        OUTPUT_DIR / "raw_duplicate_policy_verification.csv",
        index=False,
    )

    raw_verification_summary.to_csv(
        OUTPUT_DIR / "raw_duplicate_policy_summary.csv",
        index=False,
    )

    raw_history_df.to_csv(
        OUTPUT_DIR / "raw_duplicate_review_histories.csv",
        index=False,
    )

    print(
        "Raw verification outputs saved to:",
        OUTPUT_DIR,
    )

Raw verification outputs saved to: C:\Projects\gnn-recommender-dissertation\data\processed\benchmark_validation_earliest


# Key Validation Findings

This notebook validated the independently implemented preprocessing pipeline against the official **Amazon Reviews 2023 Electronics 5-core benchmark** after adopting the same duplicate-review retention policy used by the dataset authors.

The validation produced the following findings:

- **Identical aggregate statistics.** The local preprocessing pipeline reproduced the official benchmark exactly in terms of the number of interactions (15,473,536), users (1,641,026), products (368,228), and unique user–product pairs (15,473,536).

- **Identical graph structure.** A complete reconciliation of all user–product pairs showed that there were no missing or additional interactions in either dataset. The generated bipartite graph is therefore structurally identical to the official benchmark.

- **Near-complete event-level agreement.** Of the 15,473,536 shared user–product pairs, 15,473,535 (99.999994%) contained identical ratings and timestamps. Only a single user–product pair differed, with an identical timestamp but a different retained rating.

- **Verified source of the remaining difference.** Inspection of the differing record showed that both datasets retained a review with the same timestamp but different ratings, indicating a rare tie case in the raw data. This isolated difference represents the only remaining discrepancy between the independently generated dataset and the official benchmark.

- **Pipeline correctness.** Apart from this single edge case, the independently implemented preprocessing pipeline reproduces the published Amazon Reviews 2023 Electronics 5-core benchmark.

## Conclusion

The preprocessing pipeline successfully reproduces the official Amazon Reviews 2023 Electronics 5-core benchmark. The validation confirms identical dataset structure, identical graph topology, and agreement for 15,473,535 of the 15,473,536 retained interactions. The only remaining discrepancy is a single duplicate-review tie case involving identical timestamps, making the reproduced benchmark effectively equivalent to the official release.

# Generative AI use

AI assisted with refining the SQL verification query in Section 13, which was used to investigate preprocessing differences against the raw review data and the official Amazon benchmark.